##### Step 6: Feature engineering and train/test split  
6A. Create a new notebook: notebooks/05_modeling.ipynb

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/model_table.csv", parse_dates=["order_purchase_timestamp"])
df = df[df["order_purchase_timestamp"] >= "2017-01-01"].copy()
print(df.shape, df["is_late"].mean().round(3))

(96203, 31) 0.081


6B. New features (all known at purchase time)

In [2]:
df["freight_ratio"] = df["total_freight"] / df["total_price"].replace(0, np.nan)
df["is_weekend"]    = (df["purchase_weekday"] >= 5).astype(int)
df["price_per_item"] = df["total_price"] / df["n_items"]

Feature	                    Why  
__freight_ratio__	Expensive shipping relative to price often means a heavy or remote order  
__is_weekend__	Weekend orders wait for processing  
__price_per_item__	Separates bulk orders from single items

__If skipped:__ The model sees only raw columns and misses these ratios, which trees can’t easily build on their own.

6C. Remove leaky and useless columns

In [3]:
drop_cols = ["delivery_days",               # known only after delivery = LEAKAGE
             "order_id", "customer_id", 
             "customer_unique_id", 
             "seller_id",                   # IDs memorize, not generalize
             "customer_zip_code_prefix", 
             "seller_zip_code_prefix",     # too many unique values
             "cust_lat", "cust_lng", 
             "sell_lat", "sell_lng",       # distance_km already captures them
             "payment_value"]              # near-duplicate of total_price + freight
y = df["is_late"]
X = df.drop(columns=drop_cols + ["is_late"])
print(X.columns.tolist())

['order_purchase_timestamp', 'estimated_days', 'customer_state', 'n_items', 'n_sellers', 'total_price', 'total_freight', 'total_weight_g', 'total_volume_cm3', 'category', 'max_installments', 'payment_type', 'seller_state', 'distance_km', 'same_state', 'purchase_month', 'purchase_weekday', 'purchase_hour', 'freight_ratio', 'is_weekend', 'price_per_item']


Why drop IDs: The model would memorize individual rows and fail on new data.
If delivery_days stays: You get about 99% accuracy in testing, and the model is useless in real life.

6D. Time-based split

In [4]:
df_sorted = df.sort_values("order_purchase_timestamp")
cut =  int(len(df_sorted) * 0.8)
train, test = df_sorted.iloc[:cut], df_sorted.iloc[cut:]

X_train = train.drop(columns=drop_cols + ["is_late"]); y_train = train["is_late"]
X_test  = test.drop(columns=drop_cols + ["is_late"]);  y_test  = test["is_late"]

print(len(X_train), len(X_test))
print("Late rate train:", y_train.mean().round(3), "| test:", y_test.mean().round(3))
print("Test period:", test["order_purchase_timestamp"].min().date(), "to", test["order_purchase_timestamp"].max().date())

76962 19241
Late rate train: 0.088 | test: 0.053
Test period: 2018-05-27 to 2018-08-29


Why time-based, not random: In real life you predict future orders from past ones. A random split lets the model peek at the future and gives over-optimistic scores.  
Why 80/20: It leaves enough data to learn from while keeping a test set large enough to be reliable.  
__Rule:__ Never touch the test set until the final evaluation. Tuning on it is cheating.

6E. Save and commit

In [5]:
X_train.assign(is_late=y_train).to_csv("../data/processed/train.csv", index=False)
X_test.assign(is_late=y_test).to_csv("../data/processed/test.csv", index=False)

`powershell`  
git add .  
git commit -m "Step 6: feature engineering and time-based split"  
git push  

In [6]:
print(y_train.mean().round(3), y_test.mean().round(3))

0.088 0.053


First: your late rates reveal drift

Train 8.8% vs test 5.3%. This is a real shift, not a bug. Orders from late 2017 to early 2018 (Black Friday and other peaks) were delayed more often than orders from May to Aug 2018. Check it:

In [7]:
print(df.groupby(df["order_purchase_timestamp"].dt.to_period("M"))["is_late"].mean().round(3))

order_purchase_timestamp
2017-01    0.031
2017-02    0.032
2017-03    0.056
2017-04    0.079
2017-05    0.036
2017-06    0.039
2017-07    0.034
2017-08    0.033
2017-09    0.052
2017-10    0.053
2017-11    0.143
2017-12    0.084
2018-01    0.066
2018-02    0.160
2018-03    0.214
2018-04    0.053
2018-05    0.082
2018-06    0.014
2018-07    0.045
2018-08    0.104
Freq: M, Name: is_late, dtype: float64


What it means for you:

The test baseline is 5.3%, so a random model gets a PR-AUC of about 0.053, not 0.088. Always compare against the test rate.
Precision will look lower on test, because there are fewer late orders to find.  
Mention this drift in your report. It is an honest, real-world limitation.

Step 7: Preprocessing pipeline and baseline models

Why a pipeline: It learns imputation values and scaling from train only, then applies the same rules to test.  
If skipped: Filling or scaling with the full data lets test information leak into training, and your scores look better than reality.

7A. Load and separate columns